# Phase 1: AI Foundations & LLM Fundamentals
## Day 3: Class-Based Python Wrapper for LLM APIs

Welcome to Day 3. Today, we are transitioning from simple scripts to production-grade engineering. We will build a class-based Python wrapper for an LLM API that properly handles retries, rate limits, and error logging.

### Core Theory (Just-in-Time)

**Why use a wrapper class?**
When interacting with external APIs (like OpenAI, Anthropic, or local vLLM instances), you introduce network calls into your application. Network calls are inherently unreliable. APIs rate-limit you, connections drop, and services experience temporary downtime. 

A direct API call (`openai.chat.completions.create(...)`) sprinkled throughout your codebase leads to:
- **Code Duplication:** Repeating error handling logic.
- **Fragility:** A single rate limit error crashes the entire application.
- **Tight Coupling:** It becomes difficult to swap out the LLM provider later.

**How do we fix this?**
We encapsulate the API interaction within a Python class. This class is responsible for:
1. **State Management:** Holding configuration (API keys, base URLs, model selection).
2. **Resilience (Retries & Backoff):** Automatically retrying transient failures using exponential backoff.
3. **Observability:** Logging requests, successes, and failures so you can debug production issues.

We will use `tenacity` for robust retry logic and Python's built-in `logging` module for observability.

### Code Implementation

Below is a production-grade implementation of an LLM wrapper. It uses strict type hinting, docstrings, and robust error handling.

In [ ]:
import logging
from typing import Optional

from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
from openai import OpenAI, APIError, RateLimitError, APIConnectionError

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger("LLMWrapper")

class LLMClientWrapper:
    """
    A resilient wrapper for the OpenAI API that handles retries, 
    exponential backoff, and detailed logging.
    """
    
    def __init__(self, api_key: str, model: str = "gpt-3.5-turbo", max_retries: int = 3):
        """
        Initializes the LLM wrapper with configuration.

        Args:
            api_key (str): The API key for authentication.
            model (str): The specific LLM model to target.
            max_retries (int): Maximum number of retries for transient errors.
        """
        self.client = OpenAI(api_key=api_key)
        self.model = model
        self.max_retries = max_retries

    @retry(
        stop=stop_after_attempt(3),
        wait=wait_exponential(multiplier=1, min=2, max=10),
        retry=retry_if_exception_type((RateLimitError, APIConnectionError, APIError)),
        reraise=True
    )
    def _execute_call(self, prompt: str) -> str:
        """
        Internal method to execute the API call with retry logic applied.
        """
        logger.info(f"Executing generation using model: {self.model}")
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[{"role": "user", "content": prompt}]
        )
        
        content = response.choices[0].message.content
        if content is None:
            raise ValueError("Received empty response content from API.")
            
        return content

    def generate(self, prompt: str) -> Optional[str]:
        """
        Public interface to generate text from a prompt safely.

        Args:
            prompt (str): The input text to send to the LLM.

        Returns:
            Optional[str]: The generated text, or None if all attempts fail.
        """
        logger.info(f"Received prompt (length: {len(prompt)} characters)")
        try:
            result = self._execute_call(prompt)
            logger.info("Successfully generated response.")
            return result
        except RateLimitError as e:
            logger.error(f"Rate limit exceeded. Exhausted retries: {e}")
        except APIConnectionError as e:
            logger.error(f"Failed to connect to API: {e}")
        except APIError as e:
            logger.error(f"OpenAI API returned an error: {e}")
        except Exception as e:
            logger.exception(f"An unexpected error occurred during generation: {e}")
            
        return None

if __name__ == "__main__":
    # Example usage
    # wrapper = LLMClientWrapper(api_key="your_api_key_here")
    # response = wrapper.generate("Explain quantum computing in one sentence.")
    # print(response)
    pass


### Common Pitfalls (Production Gotchas)

1. **Infinite Retries:** Never use an unbounded retry loop. Always specify `stop_after_attempt` or `stop_after_delay`. Otherwise, a hard down service will lock up your application threads indefinitely.
2. **Retrying Non-Transient Errors:** Do not retry `AuthenticationError` or `BadRequestError` (e.g., sending a malformed prompt). These will fail 100% of the time regardless of how long you wait. Only retry transient errors like Rate Limits (429) or Server Errors (500s).
3. **Synchronous Blocking:** The wrapper above is synchronous. If you are building a web server (like FastAPI), this will block the event loop. In production web apps, you should use `AsyncOpenAI` and asynchronous retry decorators (`@retry` on `async def`).
4. **Logging PII:** Be extremely careful not to log user prompts if they contain Personally Identifiable Information (PII). Notice how the logger above only logs the *length* of the prompt, not the content itself.

### Practical Lab / Homework

**Your Task:**
Create a new class called `AnthropicClientWrapper` that mirrors the functionality of the `LLMClientWrapper` above, but uses the `anthropic` Python SDK. 

**Requirements:**
1. Use `anthropic.Anthropic` client.
2. Implement `tenacity` retries for Anthropic's specific exceptions (e.g., `anthropic.RateLimitError`, `anthropic.APIConnectionError`, `anthropic.InternalServerError`).
3. Include strict type hints and docstrings.
4. Add a method `generate_with_system_prompt(system: str, prompt: str) -> Optional[str]` that utilizes Anthropic's specific system prompt parameter.

Implement your solution in the cell below.

In [ ]:
import logging
from typing import Optional
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
import anthropic

class AnthropicClientWrapper:
    """
    A resilient wrapper for the Anthropic API that handles retries, 
    exponential backoff, and detailed logging.
    """
    
    def __init__(self, api_key: str, model: str = "claude-3-opus-20240229", max_retries: int = 3):
        """
        Initializes the Anthropic wrapper with configuration.

        Args:
            api_key (str): The API key for authentication.
            model (str): The specific Anthropic model to target.
            max_retries (int): Maximum number of retries for transient errors.
        """
        self.client = anthropic.Anthropic(api_key=api_key)
        self.model = model
        self.max_retries = max_retries

    @retry(
        stop=stop_after_attempt(3),
        wait=wait_exponential(multiplier=1, min=2, max=10),
        retry=retry_if_exception_type((
            anthropic.RateLimitError, 
            anthropic.APIConnectionError, 
            anthropic.InternalServerError
        )),
        reraise=True
    )
    def _execute_call(self, prompt: str, system: Optional[str] = None) -> str:
        """
        Internal method to execute the API call with retry logic applied.
        """
        logger.info(f"Executing generation using model: {self.model}")
        
        kwargs = {
            "model": self.model,
            "max_tokens": 1024,
            "messages": [{"role": "user", "content": prompt}]
        }
        if system:
            kwargs["system"] = system
            
        response = self.client.messages.create(**kwargs)
        
        content = response.content[0].text
        return content

    def generate_with_system_prompt(self, system: str, prompt: str) -> Optional[str]:
        """
        Public interface to generate text from a prompt safely, using a system prompt.

        Args:
            system (str): The system prompt to set context/behavior.
            prompt (str): The input text to send to the LLM.

        Returns:
            Optional[str]: The generated text, or None if all attempts fail.
        """
        logger.info(f"Received prompt (length: {len(prompt)}) with system prompt (length: {len(system)})")
        try:
            result = self._execute_call(prompt, system=system)
            logger.info("Successfully generated response.")
            return result
        except anthropic.RateLimitError as e:
            logger.error(f"Rate limit exceeded. Exhausted retries: {e}")
        except anthropic.APIConnectionError as e:
            logger.error(f"Failed to connect to API: {e}")
        except anthropic.InternalServerError as e:
            logger.error(f"Anthropic API returned an internal server error: {e}")
        except Exception as e:
            logger.exception(f"An unexpected error occurred during generation: {e}")
            
        return None

    def generate(self, prompt: str) -> Optional[str]:
        """
        Public interface to generate text from a prompt safely.

        Args:
            prompt (str): The input text to send to the LLM.

        Returns:
            Optional[str]: The generated text, or None if all attempts fail.
        """
        return self.generate_with_system_prompt(system="", prompt=prompt)
